# Inicialização

In [1]:
from pathlib import Path
import os
import logging
import sys

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

## ---------LOGGING-----------
# Define paths based on environment
if Path("/kaggle").exists():
    logging.info("IN KAGGLE")
    os.environ["AMBIENTE"] = "KAGGLE"
    os.environ["TENSORBOARD_NO_TF"] = "1"

    PATH_DATASET = Path("/kaggle/working/STROKE_PREDICTION")
    PATH_CODE = PATH_DATASET / "src"
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"
elif Path("/content").exists():
    os.environ["AMBIENTE"] = "COLAB"
    # PATH_DATASET = Path("/content/DELETAR")
    # PATH_CODE = PATH_DATASET / "src"
    # PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    # EXPORT_DIR = PATH_OUTPUT_DIR / "export"
    raise Exception
else:
    os.environ["AMBIENTE"] = "LOCAL"
    PATH_CODE = Path.cwd()
    PATH_DATASET = PATH_CODE
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    # Add to setup cell
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"

assert type(PATH_DATASET) is not str, "PATH_DATASET NÃO DEVE SER STR!"
# Check if installation has been done
INSTALL_MARKER = PATH_DATASET / ".install_complete"

try:
    if not INSTALL_MARKER.exists():
        # Environment-specific setup
        if os.environ["AMBIENTE"] == "KAGGLE":
            import kaggle_secrets

            os.system("pip install uv")

            user_secrets = kaggle_secrets.UserSecretsClient()
            github_pat = user_secrets.get_secret("GITHUB_PAT")

            os.chdir("/kaggle/working")
            os.system(
                f"git clone -b class-imbalance https://{github_pat}@github.com/lfaoliveira/TRAB_SERIES_TEMP.git"
            )
            os.chdir(PATH_DATASET)

        # Install dependencies
        os.chdir(PATH_DATASET)
        os.system("uv pip install --requirements pyproject.toml --system")

        if os.environ["AMBIENTE"] == "KAGGLE":
            os.system(
                "uv pip install --upgrade --force-reinstall --no-cache-dir scipy numpy matplotlib protobuf tensorboard"
            )

        # Mark installation as complete
        INSTALL_MARKER.touch()
        logging.info("Installation completed")
    else:
        logging.info("Installation already completed, skipping...")

    os.chdir(PATH_CODE)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    logging.info(f"Current working directory: {os.getcwd()}")
except Exception as e:
    logging.info("FALHA AO INICIAR NOTEBOOK")
    logging.info(e)

INFO:root:Installation already completed, skipping...
INFO:root:Current working directory: c:\Users\LUIS FELIPE\Desktop\SERIES_TEMP


## Carregamento

In [2]:
from pathlib import Path


# from src.data.dataset import NasaDataset
# from src.data.datamodule import StrokeDataModule

# ============================================================
# CARREGAMENTO E EXPLORAÇÃO DOS DADOS
# ============================================================

import traceback
from typing import Dict, Tuple

import numpy as np
import pandas as pd

from src.config.config import ProjectSettings
from darts import TimeSeries
from darts.utils.data import (
    SequentialTorchTrainingDataset,
    SequentialTorchInferenceDataset,
)
from src.data.torch_dataset import Horizons
import ast
from pandas import DataFrame


class NasaDataset:
    """Loads multivariate telemetry .npy time series and labeled anomalies for the
    NASA SMAP/MSL anomaly-detection dataset
    (https://www.kaggle.com/datasets/patrickfleith/nasa-anomaly-detection-dataset-smap-msl).

    Dataset layout expected on disk (after extracting the Kaggle zip):
        <base_path>/
            data/
                data/
                    train/   <- one .npy per channel, shape (n_timesteps, n_features)
                    test/    <- idem
            labeled_anomalies.csv

    The first column of each .npy array is the univariate target telemetry value;
    the remaining columns are one-hot encoded command / context features.

    labeled_anomalies.csv columns used:
        chan_id            – matches the .npy filename stem (e.g. "P-1")
        spacecraft         – "SMAP" or "MSL"
        anomaly_sequences  – list of [start, end] index pairs (1-based, inclusive)
        class              – anomaly type label (kept for reference, not used in training)

    Public interface (compatible with the existing datamodule):
        build_train_dataset()
        build_validation_dataset(train_dataset)
        build_test_dataset(train_dataset)

    Useful attributes after __init__:
    df_train  – long-format DataFrame with columns [series_id, time_idx, target, *features]
    df_test   – idem, plus an "anomaly" boolean column
    labels_df – parsed anomaly labels with columns
        [chan_id, spacecraft, class, seq_start, seq_end] (one row per sequence)
    """

    def __init__(self, base_path: Path, normalize: bool = False, verbose=False) -> None:
        self.verbose = verbose
        base_path.mkdir(parents=True, exist_ok=True)
        self.labels_file = base_path / "labeled_anomalies.csv"

        prototype = ProjectSettings.run_mode == "prototype"

        # Load raw numpy arrays ------------------------------------------------
        train_wide, test_wide = self.load_dataset(base_path, prototype=prototype, skip_load=True)
        # print(f"TRAIN WIDE:\n {train_wide}")
        test_wide[:10_000].to_csv("./test_wide.csv")

        # Parse labels ---------------------------------------------------------
        self.labels_df = self._read_labels()
        self.labels_df[:10_000].to_csv("./labels.csv")

        # Convert to long format -----------------------------------------------
        self.df_test = self.multi_to_df(test_wide, include_labels=True)
        self.df_test[:10_000].to_csv("./test.csv")

        # Get feature columns (all columns except time_idx, target, anomaly) ---
        self.feature_cols = [
            col for col in self.df_test.columns if col not in ["time_idx", "target"]
        ]

        # Dataset hyperparams --------------------------------------------------
        self.frequency = ProjectSettings.dataset.dataset_frequency
        self.input_width = Horizons.input_width(self.frequency)
        self.output_width = Horizons.output_width(self.frequency)
        # self.series_normalizer = GroupNormalizer(groups=["series_id"])

        # Lazily built TimeSeriesDataSet objects
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    # ------------------------------------------------------------------
    # Loading
    # ------------------------------------------------------------------

    def load_dataset(
        self, base_path: Path, prototype: bool = False, verbose: bool = False, skip_load=True
    ) -> Tuple[pd.DataFrame | None, pd.DataFrame | None]:
        """Carrega os arquivos M4 em wide format e cachea em Parquet."""
        download_path = base_path
        base_path = base_path / "data" / "data"
        train_dir = base_path / "train"
        test_dir = base_path / "test"
        pqt_train: Path = download_path / "nasa_train.parquet"
        pqt_test: Path = download_path / "nasa_test.parquet"

        # Initialize variables to avoid unbound variable errors
        df_train: DataFrame | None = None
        df_test: DataFrame | None = None

        try:
            if pqt_train.exists() and pqt_test.exists():
                # se existe, tenta ler parquet
                if self.verbose:
                    logging.info(f"Carregando cache: {pqt_train}")
                if not skip_load:
                    df_train = pd.read_parquet(pqt_train, engine="auto")
                    df_test = pd.read_parquet(pqt_test, engine="auto")
            # DOWNLOAD DO DATASET
            elif not train_dir.exists() or not test_dir.exists():
                if self.verbose:
                    logging.info(f"Baixando M4 via KaggleHub para {download_path}")
                import kagglehub

                kagglehub.dataset_download(
                    "patrickfleith/nasa-anomaly-detection-dataset-smap-msl",
                    output_dir=str(download_path),
                )

                if not train_dir.exists() or not test_dir.exists():
                    raise FileNotFoundError(f".npy DA NASA não encontrados em {base_path}")

            print("Loading train split …")
            df_train = self._load_npy_directory(train_dir, prototype=prototype)
            print(f"DF TRAIN:\n {df_train.head(5)}")
            print("Loading test split …")
            df_test = self._load_npy_directory(test_dir, prototype=prototype)

            if df_train is None or df_test is None:
                raise RuntimeError("Failed to load dataset: df_train or df_test is None")

            df_train.to_parquet(pqt_train, compression="gzip", engine="auto", index=True)
            df_test.to_parquet(pqt_test, compression="gzip", engine="auto", index=True)
            if verbose:
                logging.info(f"Cache criado: {pqt_train} e {pqt_test}")
        except Exception as e:
            if verbose:
                print(f"ERRO AO CARREGAR DATASET: {e}")
            traceback.print_exc()

        if verbose:
            logging.info(f"Dataset: treino {df_train.shape}, teste {df_test.shape}")

        return df_train, df_test

    def _load_npy_directory(self, directory: Path, prototype: bool = False) -> pd.DataFrame:
        """Read every .npy file in *directory*.

        Each file must contain an array of shape (n_timesteps,) or
        (n_features, n_timesteps).  1-D arrays are expanded to (n_timesteps, 1).
        Returns a DataFrame with MultiIndex (series_id, feature_id, time_idx ).
        """
        if not directory.exists():
            raise FileNotFoundError(f"Directory not found: {directory}")

        records = []

        for path in sorted(directory.glob("*.npy")):
            chan_id = path.stem
            arr: np.ndarray = np.load(path)
            n_timesteps, n_features = arr.shape

            # Build records with MultiIndex structure
            for f in range(n_features):
                for t in range(n_timesteps):
                    records.append(
                        {
                            "series_id": chan_id,
                            "feature_id": f"feat_{f}",
                            "time_idx": t,
                            "value": arr[t, f],
                        }
                    )

            if prototype:
                print(f"  Loaded (prototype): {chan_id}  shape={arr.shape}")
                # one channel is enough for prototyping

        # Create DataFrame with MultiIndex
        df = pd.DataFrame(records)

        # Set MultiIndex
        df = df.set_index(["series_id", "feature_id", "time_idx"])
        df.sort_index(level=0)

        print(
            f"Loaded {len(df.index.get_level_values('series_id').unique())} channels from {directory}"
        )
        return df

    # ------------------------------------------------------------------
    # Label parsing
    # ------------------------------------------------------------------

    def _read_labels(self) -> pd.DataFrame:
        """Parse labeled_anomalies.csv into a tidy DataFrame.

        Returns a DataFrame with one row per anomalous sequence:
            series_id      – channel identifier matching .npy filename stem
            spacecraft   – "SMAP" or "MSL"
            class        – anomaly type string
            seq_start    – start index (0-based, inclusive)
            seq_end      – end index   (0-based, inclusive)
        """

        df = pd.read_csv(
            self.labels_file,
            usecols=["chan_id", "anomaly_sequences"],
        )

        records = []
        for _, row in df.iterrows():
            chan_id = str(row["chan_id"]).strip()
            # anomaly_sequences is a string like "[[2149, 2349], [4536, 4844]]"
            sequences = ast.literal_eval(str(row["anomaly_sequences"]))
            for start, end in sequences:
                records.append(
                    {
                        "series_id": chan_id,
                        # The original labels are 1-based; convert to 0-based.
                        "seq_start": max(0, int(start) - 1),
                        "seq_end": max(0, int(end) - 1),
                    }
                )

        labels = pd.DataFrame(records)
        print(
            f"Loaded {len(labels)} anomaly sequences across {labels['series_id'].nunique()} channels."
        )
        return labels

    # ------------------------------------------------------------------
    # Wide dict → long DataFrame
    # ------------------------------------------------------------------

    def multi_to_df(
        self, df_multi: pd.DataFrame, include_labels: bool = False, drop=True
    ) -> pd.DataFrame:
        """Converte o DataFrame MultiIndex para o formato final desejado:

        time_idx | target | feat_1 | feat_2 | ...
        """
        if drop:
            print(df_multi.head(5))
            row_all_nan = (
                df_multi["value"].groupby(level="time_idx").transform(lambda x: x.isnull().all())
            )

            # deixa apenas os time_idx que NÃO são totalmente compostos por NaN
            df_multi = df_multi[~row_all_nan]
        # 1. Faz o unstack para trazer os 'feat_x' para as colunas
        df_wide = df_multi["value"].unstack(level="feature_id")

        # 3. Reseta o índice para mover 'series_id' e 'time_idx' de volta como colunas comuns
        df_wide = df_wide.reset_index()

        # 4. Ordena por series_id e time_idx para garantir a ordem correta
        df_wide = df_wide.sort_values(by=["series_id", "time_idx"])

        

        # 5. Se include_labels for True, cria a coluna de anomalias
        if include_labels and not self.labels_df.empty:
            # Cria uma cópia do target como base para a coluna de anomalias
            df_wide["target"] = 0

            # Processa cada série_id no labels_df
            for series_id in self.labels_df["series_id"].unique():
                # Filtra as anomalias para esta série
                series_labels = self.labels_df[self.labels_df["series_id"] == series_id]

                # Filtra o df_wide para esta série
                series_mask = df_wide["series_id"] == series_id
                series_data = df_wide[series_mask]

                # Se não houver dados para esta série, pula
                if series_data.empty:
                    raise ValueError(f"ERA PRA TER VALOR NA SERIE {series_id}")

                # Obtém o maior time_idx para esta série
                max_time_idx = series_data["time_idx"].max()

                # Cria um array booleano de zeros com o tamanho do maior time_idx + 1
                anomaly_array = np.zeros(max_time_idx + 1, dtype=int)

                # Para cada sequência de anomalia nesta série
                for _, row in series_labels.iterrows():
                    seq_start = int(row["seq_start"])
                    seq_end = int(row["seq_end"])
                    # Garante que os índices estão dentro dos limites
                    seq_start = max(0, seq_start)
                    seq_end = min(max_time_idx, seq_end)
                    # Marca os índices entre seq_start e seq_end como 1
                    anomaly_array[seq_start : seq_end + 1] = 1

                # Atualiza a coluna target no df_wide para esta série
                df_wide.loc[series_mask, "target"] = df_wide.loc[series_mask, "time_idx"].apply(
                    lambda x: anomaly_array[x] if x <= max_time_idx else np.nan
                )

        return df_wide

    # ------------------------------------------------------------------
    # TimeSeriesDataSet builders
    # ------------------------------------------------------------------

    def _build_dataset(self, data: pd.DataFrame) -> list[TimeSeries]:
        print(f"DF TEST: {self.df_test}")
        train_set = TimeSeries.from_group_dataframe(
            self.df_test,
            group_cols="series_id",
            time_col="time_idx",
            value_cols=self.feature_cols,
        )

        return train_set

    def build_train_dataset(self) -> list[TimeSeries]:
        self.train_dataset = self._build_dataset(self.df_test)
        return self.train_dataset

    def build_validation_dataset(self) -> list[TimeSeries]:
        self.val_dataset = TimeSeries.from_group_dataframe(
            self.df_test,
            group_cols="series_id",
            time_col="time_idx",
            value_cols=self.feature_cols,
        )
        return self.val_dataset

    def build_test_dataset(self, train_dataset: list[TimeSeries]) -> list[TimeSeries]:
        """Build the test TimeSeriesDataSet.

        Evaluation requires that each test window is preceded by a full encoder
        window of training data, so we concatenate the train and test portions
        per channel with re-indexed time steps.
        """
        self.test_dataset = TimeSeries.from_group_dataframe(
            self.df_test,
            group_cols="series_id",
            time_col="time_idx",
            value_cols=["target"],
        )
        return self.test_dataset


# Instanciar dataset (carrega e preprocessa automaticamente)
print("Carregando StrokeDataset...")
PATH_DATA = Path("data")

try:
    dataset = NasaDataset(PATH_DATA / "nasa", normalize=False)
    train_series = dataset.build_train_dataset()
    # print(train_set.head())
    print([elem for elem in train_series])
    print("\n✓ Dataset carregado com sucesso!")
except Exception as e:
    print(e)
    raise


Carregando StrokeDataset...
Loading train split …
  Loaded (prototype): A-1  shape=(2880, 25)
  Loaded (prototype): A-2  shape=(2648, 25)
  Loaded (prototype): A-3  shape=(2736, 25)
  Loaded (prototype): A-4  shape=(2690, 25)
  Loaded (prototype): A-5  shape=(705, 25)
  Loaded (prototype): A-6  shape=(682, 25)
  Loaded (prototype): A-7  shape=(2879, 25)
  Loaded (prototype): A-8  shape=(762, 25)
  Loaded (prototype): A-9  shape=(762, 25)
  Loaded (prototype): B-1  shape=(2435, 25)
  Loaded (prototype): C-1  shape=(2158, 55)
  Loaded (prototype): C-2  shape=(764, 55)
  Loaded (prototype): D-1  shape=(2849, 25)
  Loaded (prototype): D-11  shape=(2611, 25)
  Loaded (prototype): D-12  shape=(312, 25)
  Loaded (prototype): D-13  shape=(1490, 25)
  Loaded (prototype): D-14  shape=(3675, 55)
  Loaded (prototype): D-15  shape=(2074, 55)
  Loaded (prototype): D-16  shape=(1451, 55)
  Loaded (prototype): D-2  shape=(2880, 25)
  Loaded (prototype): D-3  shape=(2880, 25)
  Loaded (prototype): D-4 

DuplicateError: Expected unique column names, got:
- 'series_id' 2 times

In [ ]:
from darts.ad.scorers import KMeansScorer
from darts.ad.detectors import QuantileDetector
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

# =====================================================
# TRAIN
# =====================================================

scorer = KMeansScorer(
    k=8,
    window=64,
    component_wise=False,
)

scorer.fit(train_series)

# =====================================================
# TEST
# =====================================================

scores = []

for ts in test_series:
    score = scorer.score(ts)
    scores.append(score)

# =====================================================
# GLOBAL EVALUATION
# =====================================================

y_true = []
y_score = []

for labels, score in zip(test_labels, scores):

    y_true.extend(
        labels.values(copy=False).flatten()
    )

    y_score.extend(
        score.values(copy=False).flatten()
    )

auc_roc = roc_auc_score(y_true, y_score)
auc_pr = average_precision_score(y_true, y_score)

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR : {auc_pr:.4f}")

    target
0   1017.1
1   1019.3
2   1017.0
3   1019.2
4   1018.7
..     ...
88  1022.2
89  1025.3
90  1023.3
91  1035.3
92  1036.7

shape: (93, 1, 1), freq: 1, size: 744.00 B
   id                                             series
0   0  [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1....
1   1  [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1....
2   2  [[1.0], [1.0], [1.0], [1.0], [0.0], [0.0], [0....
3   3  [[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [1....
4   4  [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1....


In [ ]:
import numpy as np
import pandas as pd

array: np.ndarray = np.load("data\\nasa\\data\\data\\train\\A-4.npy")
pd.DataFrame(array).to_csv("./A-4.csv")

## Exploratory Data Analysis

In [ ]:
# ============================================================
# EXPLORAÇÃO E VISUALIZAÇÃO
# ============================================================

import numpy as np
import plotly.express as px
from statsmodels.tsa.stattools import acf, pacf
from darts.datasets.datasets import TimeSeries


# Função para calcular estatísticas de cada série
def calculate_series_stats(series: TimeSeries):
    """Calcula estatísticas para uma série temporal Darts."""
    # Converte para numpy array
    values_nan = series.values().flatten()

    values = np.array([elem for elem in values_nan if elem != np.nan])

    # Estatísticas básicas
    mean = np.mean(values)
    std = np.std(values)
    q1 = np.percentile(values, 25)
    q2 = np.percentile(values, 50)  # mediana
    q3 = np.percentile(values, 75)

    # Porcentagem de NaN
    pct_nan = (np.isnan(values).sum() / len(values)) * 100

    # Elementos acima de 3 desvios padrão
    above_3std = np.sum(np.abs(values - mean) > 3 * std)

    # ACF e PACF (até 20 lags ou menos se a série for menor)
    max_lags = min(20, len(values) // 2)
    if max_lags > 1:
        acf_values = acf(values, nlags=max_lags, fft=True)
        pacf_values = pacf(values, nlags=max_lags)
    else:
        acf_values = np.array([np.nan])
        pacf_values = np.array([np.nan])

    return {
        "mean": mean,
        "std": std,
        "q1": q1,
        "q2": q2,
        "q3": q3,
        "pct_nan": pct_nan,
        "above_3std": above_3std,
        "acf": acf_values,
        "pacf": pacf_values,
        "length": len(values),
    }


# Itera sobre todas as séries de treino
print("Calculando estatísticas para cada série...")
stats_list = []

for idx, series in enumerate(dataset.train_series):
    stats = calculate_series_stats(series)
    stats["series_id"] = idx
    stats_list.append(stats)

    if (idx + 1) % max(1, int(len(dataset.train_series) / 10)) == 0:
        print(f"Progresso: {((idx + 1) / len(dataset.train_series)) * 100:.1f}%")

print(f"Total de séries processadas: {len(stats_list)}")

# Cria DataFrame principal
df_stats = pd.DataFrame(
    [{k: v for k, v in s.items() if k not in ["acf", "pacf"]} for s in stats_list]
)

# Separa ACF e PACF em colunas
df_acf = pd.DataFrame(
    [s["acf"] for s in stats_list],
    columns=[f"acf_lag{i}" for i in range(len(stats_list[0]["acf"]))],
)
df_pacf = pd.DataFrame(
    [s["pacf"] for s in stats_list],
    columns=[f"pacf_lag{i}" for i in range(len(stats_list[0]["pacf"]))],
)

# Concatena tudo
df_stats = pd.concat([df_stats, df_acf, df_pacf], axis=1)

print(f"DataFrame de estatísticas criado: {df_stats.shape}")
print(df_stats.head())


# ============================================================
# VISUALIZAÇÕES
# ============================================================

# Heatmap: média +- desvio padrão das séries (bin=100)
# Criar bins de 100 em 100 séries
n_series = len(df_stats)
n_bins = 100
bin_size = n_series // n_bins

# Calcular média e std por bin
heatmap_data = []
for i in range(n_bins):
    start_idx = i * bin_size
    end_idx = min((i + 1) * bin_size, n_series)
    bin_series = df_stats.iloc[start_idx:end_idx]

    mean_val = bin_series["mean"].mean()
    std_val = bin_series["std"].mean()

    heatmap_data.append(
        {
            # "bin": i,
            "mean": mean_val,
            "std": std_val,
        }
    )

df_heatmap = pd.DataFrame(heatmap_data)

# Heatmap usando plotly
fig_heatmap = px.imshow(
    df_heatmap.T.values,
    labels=dict(x="Bin de séries", y="Valor", color="Valor"),
    x=[f"Bin {i}" for i in range(n_bins)],
    y=df_heatmap.columns.tolist(),
    title=f"Heatmap: Média ± Desvio Padrão das Séries (bins de {bin_size} séries)",
    color_continuous_scale="Viridis",
    aspect="auto",
)
fig_heatmap.update_layout(width=1200, height=400)
fig_heatmap.show()

# Scatter plot: % de NaN por série
fig_scatter = px.scatter(
    df_stats,
    x="series_id",
    y="pct_nan",
    title="% de NaN por Série",
    labels={"series_id": "ID da Série", "pct_nan": "% NaN"},
    hover_data=["length"],
)
fig_scatter.update_traces(marker=dict(size=4, opacity=0.6))
fig_scatter.update_layout(width=1200, height=400)
fig_scatter.show()

# Estatísticas resumidas
print("\n=== RESUMO DAS ESTATÍSTICAS ===")
print(f"Média global: {df_stats['mean'].mean():.4f}")
print(f"Desvio padrão médio: {df_stats['std'].mean():.4f}")
print(f"Séries com > 0% NaN: {(df_stats['pct_nan'] > 0).sum()}")
print(f"Máximo % NaN: {df_stats['pct_nan'].max():.1f}%")
print(f"Média de elementos acima de 3 std: {df_stats['above_3std'].mean():.2f}")

Calculando estatísticas para cada série...
Progresso: 10.0%
Progresso: 20.0%
Progresso: 30.0%
Progresso: 39.9%
Progresso: 49.9%
Progresso: 59.9%
Progresso: 69.9%
Progresso: 79.9%


c:\Users\LUIS_FELIPE\Desktop\TRAB_SERIES_TEMP\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
c:\Users\LUIS_FELIPE\Desktop\TRAB_SERIES_TEMP\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1490: ValueWarning: Matrix is singular. Using pinv.
  warnings.warn("Matrix is singular. Using pinv.", ValueWarning)
c:\Users\LUIS_FELIPE\Desktop\TRAB_SERIES_TEMP\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
c:\Users\LUIS_FELIPE\Desktop\TRAB_SERIES_TEMP\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1490: ValueWarning: Matrix is singular. Using pinv.
  warnings.warn("Matrix is singular. Using pinv.", ValueWarning)


Progresso: 89.9%
Progresso: 99.8%
Total de séries processadas: 4227
DataFrame de estatísticas criado: (4227, 51)
          mean         std       q1       q2       q3  pct_nan  above_3std  \
0  1027.719355    7.967626  1021.00  1025.80  1035.30      0.0           0   
1  2673.698925   83.464710  2595.00  2670.50  2757.30      0.0           0   
2  1039.201075   44.824805  1008.70  1042.30  1074.50      0.0           0   
3  1043.892473   85.549013  1036.00  1064.00  1079.00      0.0           3   
4  3774.831290  647.630719  3431.69  3555.81  4397.12      0.0           0   

   length  series_id  acf_lag0  ...  pacf_lag11  pacf_lag12  pacf_lag13  \
0      93          0       1.0  ...   -0.059049    0.035238   -0.156775   
1      93          1       1.0  ...    0.093669   -0.245134    0.113378   
2      93          2       1.0  ...   -0.047565   -0.048317   -0.049749   
3      93          3       1.0  ...    0.038590    0.080112    0.106673   
4      93          4       1.0  ...    0.19


=== RESUMO DAS ESTATÍSTICAS ===
Média global: 3715.2340
Desvio padrão médio: 218.2691
Séries com > 0% NaN: 0
Máximo % NaN: 0.0%
Média de elementos acima de 3 std: 0.13


# Treinamento dos modelos

## Treinando modelo simples para detecção de outliers

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# from darts.models.forecasting.baselines import NaiveMean
import pandas as pd

model = NaiveMean()


# Função que processará cada série individualmente na Thread
def process_single_series(idx, series, output_width):
    try:
        # Inicializa um modelo próprio para a thread
        # Pega os últimos 'n' valores reais da série (Ground Truth)
        # .values() no Darts retorna shape (time, dim, sample), usamos .flatten() para o Excel
        last_n_values = series[-output_width:].values().flatten().tolist()

        # Executa o backtest (retorna um objeto TimeSeries com as predições)
        # Nota: start=0.5 ou especificar o ponto de início ajuda o backtest a saber onde começar.
        # Se omitido, ele tenta computar para o maior número de pontos possível.
        pred_series = model.backtest(
            series,
            forecast_horizon=output_width,
            start=0.8,  # Começa a prever a partir dos 80% da série, ajuste se necessário
            retrain=True,
            verbose=False,
        )

        pred_values = pred_series.values().flatten().tolist()

        return {"ID": idx, "last_n": last_n_values, "preds": pred_values, "error": None}
    except Exception as e:
        # Captura falhas em séries muito curtas ou problemas no backtest
        logging.debug(e)
        return {"ID": idx, "last_n": [], "preds": [], "error": str(e)}


# --- Execução do Pool de Threads ---

output_width = dataset.output_width
total_series = len(dataset.train_series)

# Dicionário temporário para armazenar os resultados ordenados
results = []

# Recomendado usar max_workers baseado na sua CPU ou IO, ex: 4, 8, 16...
# Como NaiveMean em CPU pura no Darts não solta o GIL 100%, se o gargalo for CPU pura,
# 'ProcessPoolExecutor' seria ainda mais rápido, mas o ThreadPool já ajuda na concorrência de loops.
with ThreadPoolExecutor(max_workers=32) as executor:
    # Agenda todas as tarefas
    futures = {
        executor.submit(process_single_series, idx, series, output_width): idx
        for idx, series in enumerate(dataset.train_series)
    }

    # Processa conforme forem terminando (permite criar a barra de progresso)
    for count, future in enumerate(as_completed(futures), 1):
        res = future.result()
        results.append(res)

        # Print do progresso de 10 em 10% de conclusão do total de jobs
        if count % max(1, int(total_series / 10)) == 0 or count == total_series:
            porcentagem = (count / total_series) * 100
            print(f"Progresso: {porcentagem:.1f}% concluído ({count}/{total_series})")

# Cria o DataFrame a partir dos resultados coletados
df_results = pd.DataFrame(results)

# Ordena pelo ID original para garantir a sequência correta das séries
df_results = df_results.sort_values("ID").reset_index(drop=True)

# Salva em Excel (Convertendo as listas internas em strings para o Excel não quebrar)
df_results["last_n"] = df_results["last_n"].astype(str)
df_results["preds"] = df_results["preds"].astype(str)

df_results.to_excel("naive_preds.xlsx", index=False)
print("Arquivo 'naive_preds.xlsx' gerado com sucesso!")

In [ ]:
import gc
import logging
import os
from pathlib import Path
from typing import Literal

import mlflow
import torch
from lightning import Trainer, seed_everything
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
from mlflow.pytorch import autolog
from src.models.kan import KANSearchSpace, MyKan
from src.models.mlp import MLP, MLPSearchSpace


def get_best_hyperparameters_from_optuna(choice: str, mlf_track_uri: str) -> dict:
    """
    Reads the best run ID from the CSV file saved during Optuna training,
    fetches the run from MLflow, and returns the hyperparameters as a dict.

    Args:
        choice: Model architecture name (e.g., "MLP", "KAN")
        mlf_track_uri: MLflow tracking URI

    Returns:
        Dictionary of hyperparameters from the best Optuna run (with string keys)
    """
    return {}  # Placeholder - implement as needed


def zip_res_simplified(
    export_dir: Path,
    filename: str,
    dest_folder: Path | None = None,
    mlflow_db_path: Path | None = None,
    optuna_db_path: Path | None = None,
    mlruns_path: Path | None = None,
):
    """
    Simplified zip function that copies databases and mlruns to export_dir, then zips everything.

    Args:
        export_dir: Directory containing all files to zip (artifacts, CSVs, etc.)
        filename: Name of the output zip file
        dest_folder: Destination folder for the zip file
        mlflow_db_path: Path to mlflow.db to copy into export_dir
        optuna_db_path: Path to optuna.db to copy into export_dir
        mlruns_path: Path to mlruns folder to copy into export_dir
    """
    import shutil

    # Ensure export_dir exists
    export_dir.mkdir(parents=True, exist_ok=True)

    # Copy databases if they exist
    if mlflow_db_path and mlflow_db_path.exists():
        shutil.copy(mlflow_db_path, export_dir / mlflow_db_path.name)
        print(f"Copied {mlflow_db_path.name} to export directory")

    if optuna_db_path and optuna_db_path.exists():
        shutil.copy(optuna_db_path, export_dir / optuna_db_path.name)
        print(f"Copied {optuna_db_path.name} to export directory")

    # Copy mlruns folder if it exists
    if mlruns_path and mlruns_path.exists():
        export_mlruns = export_dir / mlruns_path.name
        if export_mlruns.exists():
            shutil.rmtree(export_mlruns)
        shutil.copytree(mlruns_path, export_mlruns)
        print(f"Copied {mlruns_path.name} folder to export directory")

    # Determine destination folder
    if dest_folder is None:
        dest_folder = Path.cwd()
    else:
        dest_folder.mkdir(parents=True, exist_ok=True)

    # Create zip file
    zip_path = dest_folder / filename.replace(".zip", "")
    shutil.make_archive(str(zip_path), "zip", export_dir)
    print(f"PATH ZIPFILE: {zip_path.with_suffix('.zip').resolve()}")


def supress_warnings():
    # Suppress specific MLflow warnings
    logging.getLogger("mlflow.utils.requirements_utils").setLevel(logging.ERROR)
    logging.getLogger("mlflow").setLevel(logging.ERROR)

    # Suppress PyTorch Lightning info messages
    logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)


def model_choice(
    CHOICE,
    INPUT_DIMS,
    N_CLASSES,
    recall_factor: list[float] | None = None,
    optuna_hyperparams: dict[str, float | int] = {},
):
    """
    Seleciona e configura o modelo baseado na escolha.

    Args:
        CHOICE: Arquitetura do modelo ("MLP" ou "KAN")
        INPUT_DIMS: Dimensionalidade da entrada
        N_CLASSES: Número de classes
        recall_factor: Fator de recall para balanceamento (opcional)
        optuna_hyperparams: Hiperparâmetros do Optuna

    Returns:
        Tuple de (model, suggested_hparams, keys)
    """
    if recall_factor is None:
        recall_factor = [1.0, 1.0]

    if CHOICE == "MLP":
        search_space = MLPSearchSpace()

        keys = search_space.Keys
        hyperparams = {
            keys.BATCH_SIZE: 32,
            keys.HIDDEN_DIMS: 512,
            keys.LR: 2e-7,
            keys.WEIGHT_DECAY: 1e-5,
            keys.BETA0: 0.99,
            keys.BETA1: 0.999,
            keys.N_LAYERS: 80,
        }

        # Map optuna hyperparams (string keys) to Keys enum
        if optuna_hyperparams:
            mapped_hyperparams = {}
            for key_enum, default_value in hyperparams.items():
                # Get the string representation of the enum key
                key_str = key_enum.value
                if key_str in optuna_hyperparams:
                    mapped_hyperparams[key_enum] = optuna_hyperparams[key_str]
                else:
                    mapped_hyperparams[key_enum] = default_value
            hyperparams = mapped_hyperparams

        print(f"MLP HYPERPARAMS: {hyperparams}")
        suggested_hparams = search_space.suggest(hyperparams)
        model = MLP(
            INPUT_DIMS,
            N_CLASSES,
            recall_factor=recall_factor,
            hyperparameters=suggested_hparams,
        )
    elif CHOICE == "KAN":
        search_space = KANSearchSpace()
        keys = search_space.Keys
        hyperparams = {
            keys.BATCH_SIZE: 32,
            keys.HIDDEN_DIMS: 100,
            keys.LR: 2e-7,
            keys.WEIGHT_DECAY: 1e-5,
            keys.BETA0: 0.99,
            keys.BETA1: 0.999,
            keys.GRID: 184,
            keys.SPLINE_POL_ORDER: 4,
        }

        # Map optuna hyperparams (string keys) to Keys enum
        if optuna_hyperparams:
            mapped_hyperparams = {}
            for key_enum, default_value in hyperparams.items():
                # Get the string representation of the enum key
                key_str = key_enum.value
                if key_str in optuna_hyperparams:
                    mapped_hyperparams[key_enum] = optuna_hyperparams[key_str]
                else:
                    mapped_hyperparams[key_enum] = default_value
            hyperparams = mapped_hyperparams

        print(f"KAN HYPERPARAMS: {hyperparams}")
        suggested_hparams = search_space.suggest(hyperparams)
        model = MyKan(
            INPUT_DIMS,
            N_CLASSES,
            recall_factor=recall_factor,
            hyperparameters=suggested_hparams,
        )
    else:
        raise ValueError("ESCOLHA DE MODELO ERRADA!")
    return model, suggested_hparams, keys


## ======================== FUNÇÃO PRINCIPAL ========================


def main(CHOICE: str):
    """
    Função principal de treinamento.

    Args:
        CHOICE: Arquitetura do modelo ("MLP" ou "KAN")
    """
    ###------SEEDS---------###
    RAND_SEED = 42
    seed_everything(RAND_SEED)
    supress_warnings()

    AMBIENTE = os.environ["AMBIENTE"]
    GPU = True if AMBIENTE in ["KAGGLE", "COLAB"] else False

    ## ---------- VARIÁVEIS DE TREINAMENTO -----------
    cpus = os.cpu_count()
    WORKERS = cpus if cpus is not None else 1
    NUM_DEVICES = 1 if GPU else 1
    NUM_NODES = 1
    BATCH_SIZE = 64
    EPOCHS = 1
    PATIENCE = int(EPOCHS * 0.6)
    ARTIFACT_PATH = EXPORT_DIR / "artifacts"
    os.makedirs(ARTIFACT_PATH, exist_ok=True)

    #### -------- VARIÁVEIS DE LOGGING ------------
    EXP_NAME = "PROD_TRAINING"
    RUN_NAME: str | None = f"PROD_{CHOICE}"
    MLF_TRACK_URI = f"sqlite:///{PATH_CODE}/mlflow.db"

    mlflow.set_tracking_uri(MLF_TRACK_URI)
    mlflow.set_experiment(EXP_NAME)
    autolog(log_models=True, checkpoint=True, exclusive=False)

    hyperparams = get_best_hyperparameters_from_optuna(CHOICE, MLF_TRACK_URI)

    ## ---------- VARIÁVEIS DO MODELO -----------
    N_CLASSES = 2

    # Preparar datamodule
    datamodule = StrokeDataModule(BATCH_SIZE, WORKERS)
    datamodule.prepare_data()
    datamodule.setup("fit")

    INPUT_DIMS = datamodule.input_dims or -1
    assert INPUT_DIMS > 0 and INPUT_DIMS is not None

    # Criar modelo
    model, _, keys = model_choice(
        CHOICE,
        INPUT_DIMS,
        N_CLASSES,
        recall_factor=None,  # Sem balanceamento específico por enquanto
        optuna_hyperparams=hyperparams,
    )

    _ = model(model.example_input_array)

    # Loop principal de treinamento
    with mlflow.start_run(run_name=RUN_NAME) as run:
        active_run_id = run.info.run_id

        mlflow_logger = MLFlowLogger(
            experiment_name=EXP_NAME,
            tracking_uri=MLF_TRACK_URI,
            log_model=True,
            run_id=active_run_id,
        )

        early_stopping = EarlyStopping(monitor="val_loss", patience=PATIENCE, mode="min")

        trainer = Trainer(
            max_epochs=EPOCHS,
            devices=NUM_DEVICES,
            accelerator="gpu" if GPU else "cpu",
            num_nodes=NUM_NODES,
            logger=mlflow_logger,
            enable_checkpointing=False,
            log_every_n_steps=1,
            callbacks=[early_stopping],
        )
        trainer.fit(model, datamodule=datamodule)
        mlflow.log_params(dict(model.hparams))

        # Test the model
        test_loader = datamodule.test_dataloader()
        trainer.test(model, dataloaders=test_loader)

        torch.cuda.empty_cache()

    return


if __name__ == "__main__":
    try:
        ARQ_TYPE = Literal["MLP", "KAN", "SVM", "XGBOOST"]  ## MODEL ARCHITECTURE
        models: list[ARQ_TYPE] = ["MLP", "KAN"]
        for choice in models:
            # trains model based on architecture
            if choice == "MLP":
                continue
            main(choice)

        NAME_RESZIP = "resultado_kaggle_stroke_normal"
        MLRUNS_FOLDER = PATH_CODE / "mlruns"
        MLF_TRACK_URI = f"sqlite:///{PATH_CODE}/mlflow.db"
        OPTUNA_DB_PATH = PATH_CODE / "optuna.db"
        ZIP_ROOT = PATH_DATASET / ".." if os.environ["AMBIENTE"] == "KAGGLE" else PATH_DATASET

        zip_res_simplified(
            export_dir=EXPORT_DIR,
            filename=NAME_RESZIP,
            dest_folder=ZIP_ROOT,
            mlflow_db_path=Path(MLF_TRACK_URI.replace("sqlite:///", "")),
            optuna_db_path=OPTUNA_DB_PATH,
            mlruns_path=MLRUNS_FOLDER,
        )
        print("\n", "=" * 60)
        print(f"RESULTADOS ZIPADOS {Path(ZIP_ROOT, NAME_RESZIP).resolve()}")
        print("=" * 60, "\n")

    except Exception as e:
        raise e
    gc.collect()

    if os.environ["AMBIENTE"] == "LOCAL":
        from view.dashboard import see_model

        see_model(PATH_DATASET / "mlflow.db", PATH_DATASET / ".." / "mlruns")

## Teste dos modelos

## Otimização de hiperparâmetros

## Teste da otimização

In [ ]:
# ## OUTPUT: MLFlow's Dashboard (Only works outside of Kaggle)
# ### Download the training results from Kaggle and paste them into a cloned folder of the repository
# # ============================================================
# # VISUALIZAÇÕES
# # ============================================================

# # Heatmap: média +- desvio padrão das séries (bin=100)
# # Criar bins de 100 em 100 séries
# n_series = len(df_stats)
# n_bins = 100
# bin_size = n_series // n_bins

# # Calcular média e std por bin
# heatmap_data = []
# for i in range(n_bins):
#     start_idx = i * bin_size
#     end_idx = min((i + 1) * bin_size, n_series)
#     bin_series = df_stats.iloc[start_idx:end_idx]

#     mean_val = bin_series["mean"].mean()
#     std_val = bin_series["std"].mean()

#     heatmap_data.append(
#         {
#             "bin": i,
#             "mean": mean_val,
#             "std": std_val,
#             "mean_minus_std": mean_val - std_val,
#             "mean_plus_std": mean_val + std_val,
#         }
#     )

# df_heatmap = pd.DataFrame(heatmap_data)

# # Heatmap usando plotly
# fig_heatmap = px.imshow(
#     df_heatmap[["mean_minus_std", "mean", "mean_plus_std"]].T.values,
#     labels=dict(x="Bin de séries", y="Valor", color="Valor"),
#     x=[f"Bin {i}" for i in range(n_bins)],
#     y=["mean - std", "mean", "mean + std"],
#     title=f"Heatmap: Média ± Desvio Padrão das Séries (bins de {bin_size} séries)",
#     color_continuous_scale="Viridis",
# )
# fig_heatmap.update_layout(width=1200, height=400)
# fig_heatmap.show()

# # Scatter plot: % de NaN por série
# fig_scatter = px.scatter(
#     df_stats,
#     x="series_id",
#     y="pct_nan",
#     title="% de NaN por Série",
#     labels={"series_id": "ID da Série", "pct_nan": "% NaN"},
#     hover_data=["length"],
# )
# fig_scatter.update_traces(marker=dict(size=4, opacity=0.6))
# fig_scatter.update_layout(width=1200, height=400)
# fig_scatter.show()

# # Estatísticas resumidas
# print("\n=== RESUMO DAS ESTATÍSTICAS ===")
# print(f"Média global: {df_stats['mean'].mean():.4f}")
# print(f"Desvio padrão médio: {df_stats['std'].mean():.4f}")
# print(f"Séries com > 0% NaN: {(df_stats['pct_nan'] > 0).sum()}")
# print(f"Máximo % NaN: {df_stats['pct_nan'].max():.1f}%")
# print(f"Média de elementos acima de 3 std: {df_stats['above_3std'].mean():.2f}")